# Final Test RFS

In [ ]:
import joblib
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, MetaEstimatorMixin, TransformerMixin
from sklearn.linear_model import Lasso

class MaskedFeatureSelector(BaseEstimator, TransformerMixin):
    """
    LASSO-based feature selector that ensures certain features are always kept.
    Features whose names contain any of the specified keywords will always be retained,
    regardless of their LASSO coefficients.

    Parameters:
    - feature_names: array-like of shape (n_features,)
        Names of the features corresponding to the columns of the input matrix X.
    - alpha: float, default=0.1
        Regularization strength for LASSO.
    - random_state: int, default=42
        Random seed for reproducibility.
    - must_keep_keywords: list of str, default=["ER", "HER2", "Gene"]
        Keywords to identify features that must always be kept.
    
    Methods:
    - fit(X, y): Fits the LASSO model and determines which features to keep
    - transform(X): Transforms the input matrix X by selecting the chosen features

    Returns:
    - Transformed feature matrix with selected features only.
    """

    def __init__(
        self,
        feature_names,
        alpha=0.1,
        random_state=42,
        must_keep_keywords=None,
    ):
        # feature_names is an array of strings corresponding to the columns of the preprocessed feature matrix.
        self.feature_names = np.array(feature_names)
        self.alpha = alpha
        self.random_state = random_state
        if must_keep_keywords is None:
            must_keep_keywords = ["ER", "HER2", "Gene"]
        self.must_keep_keywords = must_keep_keywords

    def fit(self, X, y=None):
        # Fit LASSO on the preprocessed features
        lasso = Lasso(
            alpha=self.alpha,
            max_iter=10000,
            random_state=self.random_state,
        )
        lasso.fit(X, y)

        coef_abs = np.abs(lasso.coef_)
        # Basic LASSO selection: non-zero coefficients
        mask = coef_abs != 0

        # If LASSO zeroes everything, keep all features
        if not mask.any():
            mask = np.ones_like(mask, dtype=bool)

        # Force ER / HER2 / Gene-related features to be kept
        for i, name in enumerate(self.feature_names):
            if any(keyword in name for keyword in self.must_keep_keywords):
                mask[i] = True

        self.mask = mask
        return self

    def transform(self, X):
        # Apply stored boolean mask
        return X[:, self.mask]

In [29]:
OUTPUT_FILE_PATH = "../results/RFSPrediction.csv"

# Load the model from the file
final_model = joblib.load('models/rfs_final_model.pkl')

In [30]:
# Sanity check on provided test example dataset
test_example = pd.read_csv("../data/TestDatasetExample.csv")
test_example_clean = test_example.replace(999, np.nan)

# Ensure same feature columns used during training
y_pred_example = final_model.predict(test_example_clean)

print("Example predictions preview:")
pd.Series(y_pred_example).head()
print("\nPrediction summary statistics:")
print(pd.Series(y_pred_example).describe())

Example predictions preview:

Prediction summary statistics:
count     3.000000
mean     52.874991
std       7.615001
min      45.989957
25%      48.785427
50%      51.580897
75%      56.317507
max      61.054117
dtype: float64


In [31]:
TEST_DATASET_PATH = "../data/FinalTestDataset2025.csv"
test = pd.read_csv(TEST_DATASET_PATH)

test.shape

(133, 119)

In [32]:
# Clean the test data
test_clean = test.replace(999, np.nan)

# Make predictions
y_pred = final_model.predict(test_clean)
pd.Series(y_pred).value_counts()

58.567354    1
54.837437    1
49.376238    1
69.634717    1
64.995614    1
            ..
65.682933    1
62.595042    1
54.807679    1
60.638239    1
57.722326    1
Name: count, Length: 133, dtype: int64

In [33]:
output_df = pd.DataFrame()
output_df['ID'] = test['ID']
output_df['RFS_Prediction'] = y_pred

output_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\n[SUCCESS] Predictions saved to '{OUTPUT_FILE_PATH}'")


[SUCCESS] Predictions saved to '../results/RFSPrediction.csv'
